# ViCTSD — CE + completion masking + alpha trên toàn bộ token nhãn

Mặc định **mọi token thuộc nội dung nhãn Sentiment/Topic nhận cùng trọng số lớp alpha**. Loss vẫn tính trên toàn bộ completion JSON; các token không giao vùng nhãn có alpha=1, prompt/padding giữ mask `-100`.

$$L=\frac{1}{|M|}\sum_{t\in M}a_t\,\mathrm{CE}_t,$$

với `M` là các token completion còn lại sau causal shift. **Gamma=0**, không focal modulation; không chia theo độ dài nhãn hoặc tổng alpha. Token giao biên nhãn/dấu câu cũng nhận alpha nếu tokenizer gộp chúng, vì loss được tính theo token.

Kế thừa `victsd_mask_alpha_ce_no_focal_r8_len1024.ipynb`: giữ prompt/few-shot, ghép `Title [SEP] Comment`, mapping toxicity và 10 topic, filtered train, toàn bộ validation; LoRA **r=8 / alpha=16 / dropout=0.05**, độ dài **1024**, batch **6**, accumulation **4**, **3 epochs**, LR **1e-4**, eval/save mỗi **200** bước.

Một seed mặc định **42**, đặt lại trước tạo LoRA. `ALPHA_SCOPE="all_label_tokens"` là nhánh cần thử; đổi thành `"terminal_only"` để chạy đối chứng cùng code/seed trên runtime mới. Thư mục đầu ra tự tách hai nhánh. Mỗi lần chỉ train cấu hình đang chọn.

Notebook dùng một GPU hỗ trợ bf16, kiểm tra **2 × 10 = 20 cặp nhãn** và mask/JSON của toàn bộ train/validation trước khi tải model. Dữ liệu CSV vẫn đọc từ Drive; không tải test khi train. Đánh giá F1 bằng pipeline inference ViCTSD hiện tại.


## 1. Cài thư viện và kết nối

Chạy ô cài đặt một lần. Nếu đã import thư viện trước khi cài, khởi động lại phiên Python rồi chạy từ ô kết nối. Không cài vLLM trong phiên huấn luyện.


In [ ]:
# Bộ phiên bản huấn luyện theo notebook UIT-VSFC tham khảo; không cần vLLM.
%pip install -q transformers==4.46.3 peft==0.10.0 trl==0.8.6 accelerate==0.29.3 datasets==2.21.0 pandas scikit-learn tensorboard


In [ ]:
from pathlib import Path
from google.colab import drive
from huggingface_hub import notebook_login

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

# Đăng nhập tương tác nếu cần; không lưu token trong notebook.
notebook_login()


## 2. Cấu hình chung

CSV mặc định trong `DATA_DIR`: `ViCTSD_train.csv`, `ViCTSD_valid.csv`. Các cột bắt buộc là `Title`, `Comment`, `Toxicity`, `Topic`. Chỉnh `DATA_DIR` nếu dữ liệu ở nơi khác. Đầu ra lưu riêng theo phạm vi alpha và seed, không ghi đè hoặc resume tự động.


In [ ]:
import os
import time
import json
import re
import shutil
import hashlib
import math
import importlib.metadata
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainingArguments,
    EarlyStoppingCallback, set_seed,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# Giữ cấu hình ViCTSD; chỉ đổi phạm vi gán class alpha giữa hai nhánh.
DATA_DIR = '/content/drive/MyDrive/label-focused-tuning/victsd'
TRAIN_CSV = f'{DATA_DIR}/ViCTSD_train.csv'
VAL_CSV = f'{DATA_DIR}/ViCTSD_valid.csv'
TEST_CSV = f'{DATA_DIR}/ViCTSD_test.csv'  # Chỉ để tham chiếu, không tải khi train.
SEED = 42
ALPHA_SCOPE = "all_label_tokens"  # hoặc "terminal_only" để chạy đối chứng cùng seed
if ALPHA_SCOPE not in {"all_label_tokens", "terminal_only"}:
    raise ValueError("ALPHA_SCOPE chỉ nhận all_label_tokens hoặc terminal_only.")
RUN_NAME = f"victsd_mask_alpha_ce_{ALPHA_SCOPE}_r8_len1024_seed{SEED}"
OUTPUT_DIR = f'{DATA_DIR}/{RUN_NAME}'

USE_ALPHA = True
FOCAL_GAMMA = 0.0
LOSS_NAME = 'masked_alpha_ce_no_focal'
SOURCE_NOTEBOOKS_SHA256 = {'victsd_mask_alpha_ce_no_focal_r8_len1024.ipynb': '6c79e0eb927652368c447e7fb2fb5013e107a3ef2e809b6eafb0f8f9de4842b8', 'uit_vsfc_mask_alpha_ce_all_label_tokens_r8_len1024.ipynb': '139687fbeba2cca6344352fcb34392aa62cc517e64926262e885f724c4db24f0'}
MODEL_ID = 'Viet-Mistral/Vistral-7B-Chat'
SEPARATOR = ' [SEP] '

LORA_R = 8
LORA_ALPHA = 16  # Hệ số LoRA, độc lập với class alpha của loss.
LORA_DROPOUT = 0.05
MAX_LENGTH = 1024
BATCH_SIZE = 6
GRAD_ACCUM = 4
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4
EVAL_STEPS = 200

if not USE_ALPHA or FOCAL_GAMMA != 0.0:
    raise ValueError("Thí nghiệm này yêu cầu USE_ALPHA=True và FOCAL_GAMMA=0.")

print(f'PyTorch: {torch.__version__}; CUDA: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('Notebook huấn luyện yêu cầu GPU CUDA.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('Cấu hình gốc dùng bf16; hãy chọn GPU hỗ trợ bf16 (ví dụ A100).')
if torch.cuda.device_count() != 1:
    raise RuntimeError('Hai notebook được cấu hình cho một GPU; hãy dùng cùng cấu hình cho các ablation.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

if Path(OUTPUT_DIR).exists() and (not Path(OUTPUT_DIR).is_dir() or any(Path(OUTPUT_DIR).iterdir())):
    raise FileExistsError(f'{OUTPUT_DIR} không rỗng. Đổi RUN_NAME để lưu một lần chạy mới.')
for path in (TRAIN_CSV, VAL_CSV):
    if not Path(path).is_file():
        raise FileNotFoundError(path)

# Đặt seed trước khởi tạo LoRA ở cả hai notebook.
set_seed(SEED)
print(f'Objective: {LOSS_NAME}; gamma={FOCAL_GAMMA}; class alpha={USE_ALPHA}')
print(f'Alpha scope={ALPHA_SCOPE}; seed={SEED}; output={OUTPUT_DIR}')
print(f'LoRA r={LORA_R}, lora_alpha={LORA_ALPHA}; effective batch={BATCH_SIZE * GRAD_ACCUM}')


## 3. Nhãn, prompt và dữ liệu

Lọc train bằng độ dài của toàn prompt gồm JSON. Giữ đủ validation và dừng nếu vượt max length, tránh mất câu trả lời. Tokenize với `add_special_tokens=False` ở cả bước lọc lẫn trainer vì chat template đã có special tokens. Không dùng test để tính alpha/chọn checkpoint.


In [ ]:
SENTIMENT_MAP = {0: 'Non-toxic', 1: 'Toxic'}
SENTIMENT_LABELS = ['Non-toxic', 'Toxic']
TOPIC_LABELS = [
    'KinhDoanh', 'OtoXemay', 'TheGioi', 'TheThao', 'KhoaHoc',
    'ThoiSu', 'PhapLuat', 'GiaoDuc', 'SucKhoe', 'GiaiTri',
]

SYSTEM_PROMPT = (
    'Thực hiện đồng thời hai nhiệm vụ:\n'
    '(1) Sentiment: Non-toxic hoặc Toxic.\n'
    '(2) Topic: KinhDoanh, OtoXemay, TheGioi, TheThao, KhoaHoc, '
    'ThoiSu, PhapLuat, GiaoDuc, SucKhoe, GiaiTri.\n'
    'Chỉ trả về JSON với đúng hai khóa: Sentiment và Topic. Không giải thích thêm.'
)
FEW_SHOT = (
    'Ví dụ:\n'
    'Bình luận: "Biden dẫn Trump 13 điểm [SEP] Cuộc bầu cử ở Mỹ là công khai và công bằng nhất."\n'
    'Output: {"Sentiment": "Non-toxic", "Topic": "TheGioi"}'
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


def merge_title_comment(df):
    df = df.copy()
    df['Title'] = df['Title'].fillna('').astype(str).str.strip()
    df['Comment'] = df['Comment'].fillna('').astype(str).str.strip()
    df['Text'] = df.apply(
        lambda row: f"{row['Title']}{SEPARATOR}{row['Comment']}" if row['Title'] else row['Comment'],
        axis=1,
    )
    return df


def validate_and_prepare(df, split):
    required = {'Title', 'Comment', 'Toxicity', 'Topic'}
    if not required.issubset(df.columns):
        raise ValueError(f'{split}: thiếu cột {sorted(required - set(df.columns))}')
    if df.empty:
        raise ValueError(f'{split}: dữ liệu rỗng.')
    df = df.copy()
    toxicity = pd.to_numeric(df['Toxicity'], errors='coerce')
    invalid = ~toxicity.isin(SENTIMENT_MAP)
    if invalid.any():
        raise ValueError(f'{split}: Toxicity ngoài 0/1 ở dòng {df.index[invalid].tolist()[:10]}')
    df['Toxicity'] = toxicity.astype('int64')
    df['Topic'] = df['Topic'].fillna('').astype(str).str.strip()
    invalid = ~df['Topic'].isin(TOPIC_LABELS)
    if invalid.any():
        raise ValueError(f'{split}: Topic không hợp lệ ở dòng {df.index[invalid].tolist()[:10]}')
    df = merge_title_comment(df)
    if df['Text'].eq('').any():
        raise ValueError(f'{split}: có dòng rỗng cả Title và Comment.')
    return df


def build_prompt(row):
    target_json = (
        f'{{"Sentiment": "{SENTIMENT_MAP[int(row["Toxicity"])]}", '
        f'"Topic": "{row["Topic"]}"}}'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'{FEW_SHOT}\n\nBình luận: "{row["Text"]}"'},
        {'role': 'assistant', 'content': target_json},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


df_train = validate_and_prepare(pd.read_csv(TRAIN_CSV), 'train')
df_val = validate_and_prepare(pd.read_csv(VAL_CSV), 'validation')


def prompt_lengths(prompts, batch_size=256):
    lengths = []
    for start in range(0, len(prompts), batch_size):
        encoded = tokenizer(
            prompts[start:start + batch_size], add_special_tokens=False,
            truncation=False, padding=False,
        )
        lengths.extend(len(ids) for ids in encoded['input_ids'])
    return lengths


# Chat template đã chứa special tokens: dùng add_special_tokens=False ở mọi nơi.
all_train_prompts = df_train.apply(build_prompt, axis=1).tolist()
df_train['prompt_len'] = prompt_lengths(all_train_prompts)
train_keep = df_train['prompt_len'].le(MAX_LENGTH).to_numpy()
df_train_filtered = df_train.loc[train_keep].copy()
train_prompts = [text for text, keep in zip(all_train_prompts, train_keep) if keep]
if not train_prompts:
    raise ValueError('Không còn mẫu train sau lọc độ dài.')

val_prompts = df_val.apply(build_prompt, axis=1).tolist()
val_lengths = prompt_lengths(val_prompts)
too_long_val = [i for i, length in enumerate(val_lengths) if length > MAX_LENGTH]
if too_long_val:
    raise ValueError(
        f'{len(too_long_val)} mẫu validation vượt {MAX_LENGTH} token; vị trí {too_long_val[:10]}. '
        'Dừng để tránh cắt mất JSON hoặc tự đổi split. Cần thống nhất xử lý độ dài ở mọi cấu hình đối chiếu.'
    )

train_dataset = Dataset.from_dict({'text': train_prompts})
eval_dataset = Dataset.from_dict({'text': val_prompts})
print(f'Train: {len(df_train)} -> {len(train_dataset)}; loại {len(df_train) - len(train_dataset)} mẫu dài.')
print(f'Validation giữ đủ: {len(eval_dataset)} mẫu.')
print('Phân phối Toxicity sau lọc:\n', df_train_filtered['Toxicity'].value_counts())
print('Phân phối Topic sau lọc:\n', df_train_filtered['Topic'].value_counts())
print('\nPrompt mẫu:\n', train_prompts[0])


## 4. Alpha theo span nhãn trong JSON và completion masking

Tính `N / (K * n_c)` riêng cho hai nhiệm vụ từ filtered train. Mỗi token giao vùng giá trị nhãn nhận alpha của đúng trường/lớp; các token khác giữ 1. Không lookup bằng ID tokenize nhãn đứng riêng, nên các nhãn trùng token cuối vẫn có alpha riêng.

`add_special_tokens=False` được giữ ở lọc độ dài, audit, preflight và SFTTrainer vì chat template đã có special tokens. Không packing nhiều mẫu vào một chuỗi.


In [ ]:
def balanced_class_weights(values, expected_labels):
    values = list(values)
    missing = set(expected_labels) - set(values)
    if missing:
        raise ValueError(f'Train sau lọc thiếu lớp: {sorted(missing)}')
    weights = compute_class_weight('balanced', classes=np.array(expected_labels), y=values)
    return {label: float(weight) for label, weight in zip(expected_labels, weights)}


# Chỉ filtered train được dùng để tính alpha: N / (K * n_c), riêng từng nhiệm vụ.
sent_weights = balanced_class_weights(
    df_train_filtered['Toxicity'].map(SENTIMENT_MAP), SENTIMENT_LABELS,
)
topic_weights = balanced_class_weights(df_train_filtered['Topic'], TOPIC_LABELS)
effective_sent_weights = {k: v if USE_ALPHA else 1.0 for k, v in sent_weights.items()}
effective_topic_weights = {k: v if USE_ALPHA else 1.0 for k, v in topic_weights.items()}
print('Alpha Sentiment thực dùng:', effective_sent_weights)
print('Alpha Topic thực dùng    :', effective_topic_weights)


def find_completion_label_spans(active_ids):
    """Trả về JSON và các chỉ số token tương đối phủ từng giá trị nhãn."""
    ids = tuple(int(token_id) for token_id in active_ids)
    if not ids:
        raise ValueError("Completion không có token được học.")
    decode_kwargs = dict(skip_special_tokens=False, clean_up_tokenization_spaces=False)
    decoded = tokenizer.decode(ids, **decode_kwargs)
    start = decoded.find("{")
    if start < 0 or decoded[:start].strip():
        raise ValueError("Completion sau mask phải bắt đầu bằng target JSON.")
    try:
        obj, end = json.JSONDecoder().raw_decode(decoded, start)
    except json.JSONDecodeError as exc:
        raise ValueError("Target JSON thiếu/cắt; kiểm tra MAX_LENGTH và completion mask.") from exc
    if not isinstance(obj, dict) or set(obj) != {"Sentiment", "Topic"}:
        raise ValueError(f"Target JSON phải có đúng hai khóa Sentiment và Topic: {obj}")

    # Prefix decode giúp dùng đúng token trong ngữ cảnh completion thực tế.
    boundaries = [0]
    for stop in range(1, len(ids) + 1):
        prefix = tokenizer.decode(ids[:stop], **decode_kwargs)
        if not decoded.startswith(prefix) or len(prefix) < boundaries[-1]:
            raise ValueError("Prefix decode không ổn định; không thể xác định span an toàn.")
        boundaries.append(len(prefix))

    spans = {}
    char_spans = {}
    pattern = r'"(?P<field>Sentiment|Topic)"\s*:\s*"(?P<label>[^"\\]*)"'
    for match in re.finditer(pattern, decoded[start:end]):
        field, name = match.group("field"), match.group("label")
        if field in spans or not name or obj[field] != name:
            raise ValueError("Không xác định được duy nhất giá trị nhãn hợp lệ.")
        left = start + match.start("label")
        right = start + match.end("label")
        positions = tuple(
            i for i in range(len(ids))
            if boundaries[i + 1] > boundaries[i]
            and boundaries[i + 1] > left and boundaries[i] < right
        )
        if not positions:
            raise ValueError(f"Không tìm được token cho {field}={name}.")
        if positions != tuple(range(positions[0], positions[-1] + 1)):
            raise ValueError(f"Span token không liên tục: {field}={name}.")
        spans[field] = positions
        char_spans[field] = (left, right)
    if set(spans) != {"Sentiment", "Topic"}:
        raise ValueError("Không tìm được đủ hai vùng nhãn.")
    if set(spans["Sentiment"]) & set(spans["Topic"]):
        raise ValueError("Hai trường chung một token; không thể áp hai alpha độc lập.")
    return obj, spans, char_spans


class TaskAwareAlphaCollator:
    def __init__(self, inner, sentiment_weights, topic_weights, scope="all_label_tokens"):
        if scope not in {"all_label_tokens", "terminal_only"}:
            raise ValueError(f"Phạm vi alpha không hợp lệ: {scope}")
        self.inner = inner
        self.scope = scope
        self.weights_by_field = {
            "Sentiment": {k: float(v) for k, v in sentiment_weights.items()},
            "Topic": {k: float(v) for k, v in topic_weights.items()},
        }
        self._cache = {}

    def __call__(self, features):
        batch = self.inner(features)
        labels = batch["labels"]
        alpha = torch.ones_like(labels, dtype=torch.float32)
        for row in range(labels.size(0)):
            active_idx = torch.where(labels[row] != -100)[0]
            if active_idx.numel() == 0:
                raise ValueError("Mẫu không có completion token được học.")
            # Notebook này không packing; mỗi hàng chứa một completion liên tục.
            if active_idx.numel() > 1 and not torch.all(active_idx[1:] - active_idx[:-1] == 1):
                raise ValueError("Completion mask không liên tục; không hỗ trợ packing nhiều mẫu.")
            ids = tuple(labels[row, active_idx].tolist())
            if ids not in self._cache:
                self._cache[ids] = find_completion_label_spans(ids)
            obj, spans, _ = self._cache[ids]
            for field, class_weights in self.weights_by_field.items():
                name = obj[field]
                if name not in class_weights:
                    raise ValueError(f"Thiếu alpha của {field}={name}.")
                relative = spans[field] if self.scope == "all_label_tokens" else spans[field][-1:]
                absolute = active_idx[list(relative)]
                alpha[row, absolute] = class_weights[name]
        batch["alpha_weights"] = alpha
        return batch


response_template_ids = tokenizer.encode("[/INST]", add_special_tokens=False)
completion_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids, tokenizer=tokenizer, mlm=False,
)
collator = TaskAwareAlphaCollator(
    completion_collator, effective_sent_weights, effective_topic_weights, scope=ALPHA_SCOPE,
)


In [ ]:
# Kiểm tra đủ 2 × 10 cặp bằng offsets độc lập, rồi kiểm tra toàn bộ dữ liệu.
alpha_audit = []
token_rows = {}
probe_features = []
probe_source = df_train_filtered.loc[df_train_filtered["prompt_len"].idxmin()].copy()
use_offset_audit = bool(getattr(tokenizer, "is_fast", False))

for sent, topic in product(SENTIMENT_LABELS, TOPIC_LABELS):
    probe = probe_source.copy()
    probe["Toxicity"] = next(k for k, v in SENTIMENT_MAP.items() if v == sent)
    probe["Topic"] = topic
    prompt = build_prompt(probe)
    encoded = tokenizer(prompt, add_special_tokens=False, truncation=False, return_offsets_mapping=use_offset_audit)
    offsets = encoded.pop("offset_mapping", None)
    if len(encoded["input_ids"]) > MAX_LENGTH:
        raise ValueError("Prompt audit vượt MAX_LENGTH.")
    features = {k: encoded[k] for k in ("input_ids", "attention_mask")}
    probe_features.append(features)
    original = completion_collator([features])
    batch = collator([features])
    if not torch.equal(batch["labels"], original["labels"]):
        raise ValueError("Mask completion đã bị thay đổi.")

    active = torch.where(batch["labels"][0] != -100)[0]
    active_ids = batch["labels"][0, active].tolist()
    obj, spans, char_spans = find_completion_label_spans(active_ids)
    if obj != {"Sentiment": sent, "Topic": topic}:
        raise ValueError(f"Completion audit sai nhãn: {obj}")

    expected = torch.ones_like(batch["alpha_weights"][0])
    field_audit = {}
    target_json = f'{{"Sentiment": "{sent}", "Topic": "{topic}"}}'
    target_start = prompt.rfind(target_json)
    if target_start < 0:
        raise ValueError("Không tìm thấy target JSON trong prompt audit.")
    for field, class_weights in [("Sentiment", effective_sent_weights), ("Topic", effective_topic_weights)]:
        positions = list(spans[field])
        absolute = active[positions].tolist()
        name = obj[field]
        if use_offset_audit:
            # Xác định khoảng nhãn từ target đã biết, độc lập với prefix parser.
            marker = f'"{field}": "'
            left = target_start + target_json.index(marker) + len(marker)
            right = left + len(name)
            by_offset = [i for i, (a, b) in enumerate(offsets) if b > a and b > left and a < right]
            if absolute != by_offset:
                raise ValueError(f"Span {field} khác offset_mapping: {absolute} != {by_offset}")
        selected = absolute if ALPHA_SCOPE == "all_label_tokens" else absolute[-1:]
        expected[selected] = float(class_weights[name])
        ids = [active_ids[i] for i in positions]
        pieces = tokenizer.convert_ids_to_tokens(ids)
        decoded_span = tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
        if name not in decoded_span:
            raise ValueError(f"Span decode không phủ đủ nhãn {field}={name}: {decoded_span!r}")
        token_rows[(field, name)] = {
            "field": field, "label": name, "n_tokens_in_json": len(ids),
            "tokens": " | ".join(pieces), "alpha": float(class_weights[name]),
            "n_alpha_positions": len(selected),
        }
        field_audit[field] = {
            "label": name, "relative_positions": positions,
            "absolute_positions": absolute, "token_ids": ids, "tokens": pieces,
            "weighted_absolute_positions": selected, "alpha": float(class_weights[name]),
        }
    if not torch.allclose(batch["alpha_weights"][0], expected, rtol=0, atol=1e-6):
        raise ValueError(f"Alpha sai ở cặp ({sent}, {topic}).")
    alpha_audit.append({"Sentiment": sent, "Topic": topic, "fields": field_audit})

# Kiểm tra cả batch nhiều độ dài: prompt/padding vẫn -100, alpha padding vẫn 1.
original_padded = completion_collator(probe_features)
padded = collator(probe_features)
assert torch.equal(padded["labels"], original_padded["labels"])
padding = padded["attention_mask"] == 0
assert torch.all(padded["labels"][padding] == -100)
assert torch.all(padded["alpha_weights"][padding] == 1)
assert len(alpha_audit) == len(SENTIMENT_LABELS) * len(TOPIC_LABELS) == 20

print(pd.DataFrame(token_rows.values()).to_string(index=False))
print(f"✅ Đủ {len(alpha_audit)}/20 cặp; mask và alpha đúng; scope={ALPHA_SCOPE}.")
print(f"   Kiểm tra offset độc lập: {use_offset_audit}; gamma={FOCAL_GAMMA}; seed={SEED}")


# Kiểm tra batch có padding: trọng số của mẫu không đổi khi ghép batch.
mixed_batch = collator(probe_features)
for row, feature in enumerate(probe_features):
    single = collator([feature])
    length = len(feature['input_ids'])
    torch.testing.assert_close(mixed_batch['labels'][row, :length], single['labels'][0])
    torch.testing.assert_close(mixed_batch['alpha_weights'][row, :length], single['alpha_weights'][0])
    if (mixed_batch['labels'][row, length:] != -100).any().item():
        raise ValueError('Padding chưa được mask.')

# Kiểm tra tất cả dữ liệu trước khi tải model 7B: không có mẫu bị mất JSON vì masking.
for split_name, dataset in (('train', train_dataset), ('validation', eval_dataset)):
    for start in range(0, len(dataset), 64):
        texts = dataset[start:start + 64]['text']
        encoded = tokenizer(texts, add_special_tokens=False, truncation=False, padding=False)
        features = [{key: value[i] for key, value in encoded.items()} for i in range(len(texts))]
        collator(features)
    print(f'{split_name}: đã kiểm tra mask/JSON cho {len(dataset)} mẫu.')

print(f'Audit thành công: {len(alpha_audit)}/20 cặp nhãn và batch có padding.')
example = collator([probe_features[0]])
print('Completion thực sự được học:')
print(tokenizer.decode(example['labels'][0][example['labels'][0] != -100].tolist()))


## 5. CE và kiểm tra số học

Nhánh train này cố định gamma=0: mean alpha-weighted CE trên shifted completion tokens, gồm cả cú pháp JSON. Giữ hàm loss và kiểm tra số học từ ViCTSD nguồn; việc kiểm tra thêm gamma=2 trong tensor nhỏ không bật focal trong huấn luyện.


In [ ]:
def masked_token_loss(logits, labels, alpha_weights, gamma):
    """Mean trên mọi token completion sau causal shift, kể cả cú pháp JSON."""
    if gamma < 0:
        raise ValueError('gamma phải không âm.')
    if alpha_weights.shape != labels.shape:
        raise ValueError('alpha_weights và labels phải cùng shape.')
    shifted_labels = labels[..., 1:]
    keep = shifted_labels != -100
    if not keep.any().item():
        raise ValueError('Batch không có token completion sau causal shift.')

    # Chọn token có loss trước khi đổi float32 để giảm bộ nhớ trung gian.
    selected_logits = logits[..., :-1, :][keep].float()
    targets = shifted_labels[keep]
    ce = F.cross_entropy(selected_logits, targets, reduction='none')
    alpha = alpha_weights[..., 1:][keep].to(device=ce.device, dtype=ce.dtype)
    if gamma == 0.0:
        return (alpha * ce).mean()  # CE có alpha, không có focal modulation.
    pt = torch.exp(-ce)
    return (alpha * (1.0 - pt).pow(gamma) * ce).mean()


class AblationTrainer(SFTTrainer):
    def __init__(self, *args, focal_gamma, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_gamma = float(focal_gamma)
        # Giữ cách lấy mean theo batch/token của code tham khảo.
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if 'alpha_weights' not in inputs or 'labels' not in inputs:
            raise ValueError('Batch thiếu labels/alpha_weights; kiểm tra data collator.')
        # Loss được tính một lần ở đây; không nhờ model tính CE mặc định.
        model_inputs = {key: value for key, value in inputs.items() if key not in {'labels', 'alpha_weights'}}
        outputs = model(**model_inputs)
        loss = masked_token_loss(
            outputs.logits, inputs['labels'], inputs['alpha_weights'], self.focal_gamma,
        )
        outputs['loss'] = loss
        return (loss, outputs) if return_outputs else loss


# Kiểm tra số học trên tensor nhỏ: causal shift, mask, alpha và gradient.
def check_loss_math():
    logits = torch.linspace(-2.0, 2.0, 2 * 5 * 7).reshape(2, 5, 7).requires_grad_()
    labels = torch.tensor([[-100, -100, 2, 3, -100], [-100, 1, 4, -100, -100]])
    ones = torch.ones_like(labels, dtype=torch.float32)
    weights = ones.clone()
    weights[0, 2] = 3.0
    weights[1, 2] = 0.5
    for gamma in (0.0, 2.0):
        for alpha in (ones, weights):
            terms = []
            for row in range(labels.size(0)):
                for target_pos in range(1, labels.size(1)):
                    target = int(labels[row, target_pos])
                    if target == -100:
                        continue
                    ce = -F.log_softmax(logits[row, target_pos - 1].float(), dim=-1)[target]
                    terms.append(alpha[row, target_pos] * (1.0 - torch.exp(-ce)).pow(gamma) * ce)
            actual = masked_token_loss(logits, labels, alpha, gamma)
            torch.testing.assert_close(actual, torch.stack(terms).mean())
    changed = logits.detach().clone()
    changed[0, 0, :] = 100.0  # Chỉ dự đoán nhãn đã mask.
    torch.testing.assert_close(
        masked_token_loss(changed, labels, weights, 2.0),
        masked_token_loss(logits, labels, weights, 2.0),
    )
    masked_token_loss(logits, labels, weights, FOCAL_GAMMA).backward()
    if not torch.isfinite(logits.grad).all().item() or logits.grad.abs().sum().item() == 0:
        raise AssertionError('Gradient không hợp lệ.')
    if logits.grad[0, 0].abs().sum().item() != 0:
        raise AssertionError('Token đã mask vẫn có gradient.')
    try:
        masked_token_loss(logits, torch.full_like(labels, -100), ones, 2.0)
    except ValueError:
        pass
    else:
        raise AssertionError('Batch không có completion phải báo lỗi.')
    print('Loss đã qua kiểm tra số học, mask, causal shift và gradient.')


check_loss_math()


## 6. Model, LoRA và trainer

Tải model sau khi audit và preflight dữ liệu thành công. Đặt lại seed trước LoRA; giữ cấu hình ViCTSD một GPU, `use_reentrant=False`, workers=2 và 3 epochs. `label_names=['labels']` bảo đảm evaluation đi qua custom loss; `prediction_loss_only=True` tránh lưu toàn bộ logits khi eval.


In [ ]:
print('Đang tải Vistral-7B bằng bf16, không quantize...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Đặt lại ngay trước LoRA để các kiểm tra trước đó không làm lệch RNG giữa hai run.
set_seed(SEED)
peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none', task_type='CAUSAL_LM',
)
model_with_lora = get_peft_model(model, peft_config)
model_with_lora.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    optim='adamw_torch_fused', learning_rate=LEARNING_RATE,
    weight_decay=0.01, max_grad_norm=1.0,
    lr_scheduler_type='cosine', warmup_ratio=0.05,
    bf16=True, fp16=False, tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    evaluation_strategy='steps', eval_steps=EVAL_STEPS,
    save_strategy='steps', save_steps=EVAL_STEPS, save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss', greater_is_better=False,
    logging_steps=10, report_to='tensorboard',
    group_by_length=True,
    dataloader_num_workers=2, dataloader_pin_memory=True,
    seed=SEED, data_seed=SEED,
    label_names=['labels'], prediction_loss_only=True,
)

trainer = AblationTrainer(
    model=model_with_lora,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    tokenizer=tokenizer, args=training_args,
    dataset_text_field='text', max_seq_length=MAX_LENGTH, packing=False,
    dataset_kwargs={'add_special_tokens': False},
    data_collator=collator, focal_gamma=FOCAL_GAMMA,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4, early_stopping_threshold=0.001)],
)

# Đối chiếu tokenizer của SFTTrainer với tokenizer dùng lọc/audit.
for dataset_name, actual_dataset, source_prompts in (
    ('train', trainer.train_dataset, train_prompts),
    ('validation', trainer.eval_dataset, val_prompts),
):
    for index in sorted({0, len(actual_dataset) - 1}):
        expected_ids = tokenizer(source_prompts[index], add_special_tokens=False)['input_ids']
        if actual_dataset[index]['input_ids'] != expected_ids:
            raise ValueError(f'{dataset_name}: tokenization của trainer khác bước tiền xử lý.')

steps_per_epoch = math.ceil(math.ceil(len(train_dataset) / BATCH_SIZE) / GRAD_ACCUM)
estimated_steps = steps_per_epoch * NUM_EPOCHS
if estimated_steps < EVAL_STEPS:
    raise ValueError('Không đủ bước để có validation checkpoint; thống nhất eval/save_steps ở mọi cấu hình.')
print(f'Số bước dự kiến khoảng {estimated_steps}; eval/save mỗi {EVAL_STEPS} bước.')
print(f'VRAM allocated trước train: {torch.cuda.memory_allocated() / 1e9:.1f} GB')


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def frame_sha256(frame):
    payload = frame[['Title', 'Comment', 'Text', 'Toxicity', 'Topic']].to_json(
        orient='split', force_ascii=False,
    )
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


manifest = {
    'run_name': RUN_NAME, 'status': 'ready', 'dataset': 'ViCTSD', 'model_id': MODEL_ID,
    'notebook': 'victsd_mask_alpha_ce_all_label_tokens_r8_len1024.ipynb',
    'reference_notebook': 'victsd_mask_alpha_ce_no_focal_r8_len1024.ipynb',
    'source_notebooks_sha256': SOURCE_NOTEBOOKS_SHA256,
    'model_resolved_revision': getattr(model.config, '_commit_hash', None),
    'tokenizer_resolved_revision': tokenizer.init_kwargs.get('_commit_hash'),
    'tokenizer_backend_sha256': (
        hashlib.sha256(tokenizer.backend_tokenizer.to_str().encode('utf-8')).hexdigest()
        if getattr(tokenizer, 'is_fast', False) else None
    ),
    'loss': {
        'name': LOSS_NAME, 'completion_mask': True,
        'gamma': FOCAL_GAMMA, 'alpha_balancing': USE_ALPHA,
        'alpha_assignment': 'json_field_and_actual_completion_span',
        'alpha_scope': ALPHA_SCOPE,
        'label_length_normalization': False,
        'offset_audit': use_offset_audit,
        'reduction': 'mean_over_unmasked_shifted_completion_tokens',
        'other_completion_tokens_alpha': 1.0,
    },
    'lora': {'r': LORA_R, 'lora_alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT},
    'max_length': MAX_LENGTH, 'seed_before_lora': SEED,
    'add_special_tokens': False, 'packing': False,
    'train_rows_raw': len(df_train), 'train_rows_filtered': len(df_train_filtered),
    'train_rows_removed_for_length': len(df_train) - len(df_train_filtered),
    'validation_rows': len(df_val),
    'train_filtered_sha256': frame_sha256(df_train_filtered),
    'validation_sha256': frame_sha256(df_val),
    'data_files': {'train': TRAIN_CSV, 'validation': VAL_CSV},
    'data_file_sha256': {'train': file_sha256(TRAIN_CSV), 'validation': file_sha256(VAL_CSV)},
    'test_used_during_training': False,
    'balanced_sentiment_weights_from_train': sent_weights,
    'balanced_topic_weights_from_train': topic_weights,
    'effective_sentiment_weights': effective_sent_weights,
    'effective_topic_weights': effective_topic_weights,
    'alpha_audit_20_pairs': alpha_audit,
    'label_token_audit': list(token_rows.values()),
    'system_prompt': SYSTEM_PROMPT, 'few_shot': FEW_SHOT,
    'response_template_ids': response_template_ids,
    'training_arguments': json.loads(training_args.to_json_string()),
    'checkpoint_selection': 'minimum_validation_loss_of_this_objective',
    'runtime_versions': {
        name: importlib.metadata.version(name)
        for name in ['torch', 'transformers', 'tokenizers', 'peft', 'trl', 'accelerate', 'datasets', 'numpy', 'pandas', 'scikit-learn']
    },
}


def save_manifest():
    with open(Path(OUTPUT_DIR) / 'run_manifest.json', 'w', encoding='utf-8') as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)


Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
save_manifest()
print('Đã lưu cấu hình, hash dữ liệu, alpha và audit tại:', OUTPUT_DIR)


## 7. Huấn luyện và lưu

Ô tiếp theo bắt đầu train. `best_model` là adapter đã chọn theo validation loss. `second_best_model` chỉ được tạo nếu checkpoint hạng hai còn trên đĩa. `last_checkpoint_model` là checkpoint lưu cuối theo lịch 200 bước, có thể sớm hơn bước kết thúc train; bước thực tế được ghi vào manifest.


In [ ]:
print(f'Bắt đầu huấn luyện {RUN_NAME}...')
manifest['status'] = 'training'
save_manifest()
start_time = time.time()
try:
    train_result = trainer.train()
except Exception as error:
    manifest.update(status='failed', error_type=type(error).__name__, error=str(error))
    save_manifest()
    raise
total_time = time.time() - start_time
trainer.save_state()

if trainer.state.best_model_checkpoint is None:
    raise RuntimeError('Chưa có best checkpoint theo validation; không đặt tên best_model cho model chưa được chọn.')

# load_best_model_at_end=True: adapter hiện tại là best theo objective của run này.
best_dir = Path(OUTPUT_DIR) / 'best_model'
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(best_dir)
saved_models = {'best_model': str(best_dir)}

# Mỗi step chỉ xếp hạng một lần. Second-best có thể đã bị checkpoint rotation xóa.
eval_by_step = {
    int(log['step']): log for log in trainer.state.log_history
    if 'eval_loss' in log and 'step' in log and math.isfinite(float(log['eval_loss']))
}
ranked_logs = sorted(eval_by_step.values(), key=lambda log: (log['eval_loss'], log['step']))
if len(ranked_logs) >= 2:
    second_step = int(ranked_logs[1]['step'])
    second_src = Path(OUTPUT_DIR) / f'checkpoint-{second_step}'
    if second_src.is_dir():
        second_dir = Path(OUTPUT_DIR) / 'second_best_model'
        shutil.copytree(second_src, second_dir)
        tokenizer.save_pretrained(second_dir)
        saved_models['second_best_model'] = str(second_dir)
    else:
        print(f'Checkpoint hạng hai (step {second_step}) đã bị rotation xóa; không tạo second_best_model.')

available_steps = [
    int(path.name.split('-')[-1]) for path in Path(OUTPUT_DIR).iterdir()
    if path.is_dir() and re.fullmatch(r'checkpoint-\d+', path.name)
]
last_saved_step = max(available_steps) if available_steps else None
if last_saved_step is not None:
    last_src = Path(OUTPUT_DIR) / f'checkpoint-{last_saved_step}'
    last_dir = Path(OUTPUT_DIR) / 'last_checkpoint_model'
    shutil.copytree(last_src, last_dir)
    tokenizer.save_pretrained(last_dir)
    saved_models['last_saved_checkpoint'] = str(last_dir)

manifest.update({
    'status': 'completed', 'train_seconds': total_time,
    'final_global_step': int(trainer.state.global_step),
    'best_model_checkpoint': trainer.state.best_model_checkpoint,
    'best_validation_loss': trainer.state.best_metric,
    'last_saved_checkpoint_step': last_saved_step,
    'saved_models': saved_models,
    'eval_logs_sorted_by_objective_loss': ranked_logs,
    'train_metrics': train_result.metrics,
})
save_manifest()

print(f'Hoàn tất sau {total_time / 3600:.2f} giờ.')
print(f'Best checkpoint: {trainer.state.best_model_checkpoint}')
print(f'Best validation objective loss: {trainer.state.best_metric}')
print(f'Bước kết thúc: {trainer.state.global_step}; checkpoint lưu cuối: {last_saved_step}')
for name, folder in saved_models.items():
    print(f'{name}: {folder}')
print('Đã lưu trainer_state.json và run_manifest.json.')


## Dùng kết quả thí nghiệm

Mặc định `class alpha=True`, `gamma=0`, `ALPHA_SCOPE="all_label_tokens"`. Dùng `OUTPUT_DIR + "/best_model"` cho inference khi đối chứng cũng chọn best theo validation loss. Giữ cùng prompt, split, decoder, parser và cách xử lý JSON không hợp lệ; so macro-F1 Sentiment/Topic, F1 từng lớp và tỷ lệ JSON hợp lệ. Chọn cấu hình bằng validation, giữ test cho đánh giá cuối.

Để chạy đối chứng mới cùng code/seed, đổi `ALPHA_SCOPE="terminal_only"` rồi chạy từ đầu trên runtime mới. Không so trực tiếp trị số `eval_loss` giữa hai cách gán alpha để kết luận chất lượng phân loại, vì hai objective khác nhau. Mẫu số vẫn là số shifted completion token được học, không phải tổng alpha hay số token nhãn.

Giữ cách lưu của ViCTSD nguồn: `best_model`, `second_best_model` nếu checkpoint còn tồn tại, `last_checkpoint_model` theo checkpoint lưu cuối. Manifest ghi phạm vi alpha, seed, hash CSV/filtered data/tokenizer, class weights, audit đủ 20 cặp, cấu hình thực tế và trạng thái train.

Notebook cần CSV train/validation trên Drive khi chạy Colab. CSV thực tế và huấn luyện model 7B chưa được kiểm tra trong lúc tạo file; các ô kiểm tra/audit sẽ chạy trên dữ liệu của bạn trước khi tải model.
